In [1]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_parquet("../data/processed/merged_protein_dataset.parquet")

In [3]:
df.shape

(2058302, 8)

In [5]:
df.head(20)

,source,id,organism,taxonomy_id,sequence,disorder_mask,coverage,bfactor
0,disprot,A0A024B7W1,Zika virus (isolate ZIKV/Human/French Polynesi...,2043570,MKNPKKKSGGFRIVNMLKRGVARVSPFGGLKRLPAGLLLGHGPIRM...,0000000000000000000000000000000000000000000000...,NaN,None
1,disprot,A0A078IB52,Brassica napus,3708,MQYYETREKEYYDVAQGQSRQSYGQNHQGYGQSQSRPVYGNSPTLN...,1111111111111111111111111111111111111111111111...,NaN,None
2,disprot,A0A096W0X9,Influenza A virus,1402874,MSLLTEVETPTRSEWECRCSDSSDPLVIAANIIGILHLILWITDRL...,0000000000000000000000000000000000000000000111...,NaN,None
3,disprot,A0A097F4I8,Lassa mammarenavirus,11620,MGNKQVKAPEARNSPRASLIPDATHLGPQFCKSCWFENKGLVECNN...,1111111111111111111111110000000000000000000000...,NaN,None
4,disprot,A0A0B4JDC9,Drosophila melanogaster,7227,MHLCEFPSANVEEENRRPEKAAAAASKKQKHKQQKSRPRGSHSMPY...,0000000000000000000000000000000000000000000000...,NaN,None
5,disprot,A0A0C4DGY3,Homo sapiens,9606,MSSEMEPLLLAWSYFRRRKFQLCADLCTQMLEKSPYDQAAWILKAR...,0000000000000000000000000000000000000000000000...,NaN,None
6,disprot,A0A0G2JXC5,Rattus norvegicus,10116,MGGVTITKAKVDFSSVVCLPPSVIAVNGLDGGGAGENDEEPVLLSL...,0000000000000000000000000000000000000000000000...,NaN,None
7,disprot,A0A0G2KTI4,Danio rerio,7955,MSASPPISAGDYLSAPEPDALKPAGPTPSQSRFQVDLVTESAGDGE...,1111111111111111111111111111111111111111111111...,NaN,None
8,disprot,A0A0H2UWN8,Streptococcus pyogenes serotype M3 (strain ATC...,198466,MLYIDEFKEAIDKGYILGDTVAIVRKNGKIFDYVLPHEKVRDDEVV...,1111111111111111111111111111111111111111111111...,NaN,None
9,disprot,A0A0H2XCS3,Xanthomonas campestris pv. campestris (strain ...,314565,MSTATNPLDLDVCAREPIHIPGLIQPYGVLLVIDPADGRIVQASTT...,0000000000000000000000000000000000000000000000...,NaN,None


In [19]:
df.info()

<class 'pandas.DataFrame'>
Index: 2054329 entries, 0 to 2058301
Data columns (total 8 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   source         str   
 1   id             str   
 2   organism       str   
 3   taxonomy_id    str   
 4   sequence       str   
 5   disorder_mask  str   
 6   coverage       str   
 7   bfactor        object
dtypes: object(1), str(7)
memory usage: 1.8+ GB


In [ ]:
str_cols = df.select_dtypes(include=["string"]).columns
df[str_cols] = df[str_cols].replace("", pd.NA)

df["bfactor"] = df["bfactor"].apply(
    lambda x: pd.NA if isinstance(x, (list, np.ndarray)) and len(x) == 0 else x
)

In [ ]:
df["organism"] = df["organism"].apply(lambda x: x.lower() if isinstance(x, str) else x)

df["sequence"] = df["sequence"].apply(lambda x: x.upper() if isinstance(x, str) else x)

In [ ]:
df = df.dropna(subset=["sequence", "taxonomy_id"], how="any")
df = df.dropna(subset=["disorder_mask", "bfactor"], how="all")

In [ ]:
def to_hashable(x):
    if isinstance(x, np.ndarray):
        return tuple(x.tolist())
    if isinstance(x, list):
        return tuple(x)
    return x


df["bfactor"] = df["bfactor"].apply(to_hashable)

In [36]:
df.shape

(2053411, 8)

In [ ]:
df = df.drop_duplicates(
    subset=["taxonomy_id", "sequence", "disorder_mask", "bfactor"], keep="first"
)

In [39]:
df.shape

(2007214, 8)

In [ ]:
duplicates = df[df.duplicated(subset=["sequence", "taxonomy_id"], keep=False)]

In [41]:
duplicates.shape

(407264, 8)

In [ ]:
duplicates.sort_values(by=["sequence", "taxonomy_id"]).head(20)

,source,id,organism,taxonomy_id,sequence,disorder_mask,coverage,bfactor
1577771,rcsb-pdb,4d2d,synthetic construct,32630,AAA,NaN,111,"(82.31999969482422, 74.7300033569336, 71.55000..."
1597165,rcsb-pdb,3e4a,synthetic construct,32630,AAA,NaN,111,"(58.959999084472656, 44.0, 51.529998779296875)"
1557137,rcsb-pdb,4c2c,escherichia coli,511693,AAA,NaN,111,"(66.2699966430664, 47.88999938964844, 43.04000..."
1557145,rcsb-pdb,4c2f,escherichia coli,511693,AAA,NaN,111,"(68.94999694824219, 65.80999755859375, 65.5899..."
1590272,rcsb-pdb,6dqq,synthetic construct,32630,AAAA,NaN,1111,"(28.709999084472656, 28.229999542236328, 36.20..."
1724069,rcsb-pdb,4ka7,synthetic construct,32630,AAAA,NaN,1111,"(49.84000015258789, 44.72999954223633, 44.8100..."
1618042,rcsb-pdb,6f5p,homo sapiens,9606,AAAAAAAAAA,NaN,1111111111,"(30.0, 30.0, 30.0, 132.39999389648438, 30.0, 3..."
1942683,rcsb-pdb,4uyz,homo sapiens,9606,AAAAAAAAAA,NaN,1111111111,"(72.30999755859375, 68.68000030517578, 68.2799..."
1992346,rcsb-pdb,3wsz,homo sapiens,9606,AAAAAAAAAA,NaN,0001111111,"(nan, nan, nan, 131.02999877929688, 133.940002..."
2005041,rcsb-pdb,6xi2,homo sapiens,9606,AAAAAAAAAA,NaN,1111111111,"(69.95999908447266, 64.13999938964844, 49.0, 4..."
